# Imports

In [3]:
# Standard Library Imports
from datetime import datetime, time
import random
# Third Party Imports
import numpy as np
# Local Imports
from generator import Generator, Distribution

# Defining Distributions

In [4]:

class MaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease A", "Disease B", "Disease C", "Disease D"]
        self.probabilities = [0.1, 0.2, 0.3, 0.4]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class FemaleDiseasesDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["Disease F", "Disease G", "Disease H", "Disease I"]
        self.probabilities = [0.4, 0.3, 0.2, 0.1]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class GenderDistribution(Distribution):
    def __init__(self):
        super().__init__()
        self.values = ["male", "female"]
        self.probabilities = [0.5, 0.5]

    def sample(self):
        return str(np.random.choice(self.values, p=self.probabilities))


class NumberOfPatientsDuringADayDistribution(Distribution):
    def __init__(self):
        super().__init__()

    def sample(self):
        return int(np.random.normal(50, 10))



class ArrivalTime(Distribution):
    def __init__(self):
        super().__init__()
        self.intervals = [(8, 12), (12, 16), (16, 20), (20, 24)]
        self.probabilities = [0.1, 0.3, 0.4, 0.2]

    def sample(self) -> str:
        start, end = random.choices(self.intervals, weights=self.probabilities, k=1)[0]
        sampled = np.random.uniform(start, end)
        h, rem = divmod(sampled * 3600, 3600)
        m, s = divmod(rem, 60)
        return time(int(h), int(m), int(s)).strftime("%H:%M:%S")
    
# Instantiate the distributions
num_patients_per_day_dist: Distribution = NumberOfPatientsDuringADayDistribution()
male_diseases_dist: Distribution = MaleDiseasesDistribution()
female_diseases_dist: Distribution = FemaleDiseasesDistribution()
gender_dist: Distribution = GenderDistribution()
arrival_time: Distribution = ArrivalTime()

# Make Distribution Tree

In [5]:
DistributionTreeNoPatientsPrDay = {
    "number_of_patients": {
        "dist": num_patients_per_day_dist,
        "no_sample_aggregation": lambda x: x,
        "arrival_time": {
            "dist": arrival_time,
            "no_sample_aggregation": lambda x: x[0],
        },
        "gender": {
            "dist": gender_dist,
            "no_sample_aggregation": lambda x: x[0],
            "disease": {
                "dist": {
                    "male": male_diseases_dist,
                    "female": female_diseases_dist
                },
                "no_sample_aggregation": len,
            }
    }
    },
}

# Instantiate and run the generator

In [6]:
generator = Generator(
    distribution_tree=DistributionTreeNoPatientsPrDay, 
    output_format=("arrival_time", "gender", "disease"), 
    output_sorting_function=lambda tup: datetime.strptime(tup[0], "%H:%M:%S").time()
)
output = generator.generate_data()
print(output)

[('08:41:18', 'male', 'Disease D'), ('09:28:11', 'female', 'Disease G'), ('09:44:57', 'male', 'Disease D'), ('10:30:24', 'male', 'Disease D'), ('12:10:45', 'female', 'Disease F'), ('12:10:49', 'female', 'Disease H'), ('12:38:05', 'male', 'Disease C'), ('12:40:58', 'female', 'Disease G'), ('12:43:29', 'male', 'Disease C'), ('12:46:07', 'male', 'Disease B'), ('13:12:20', 'male', 'Disease C'), ('13:16:22', 'male', 'Disease B'), ('13:46:11', 'female', 'Disease H'), ('13:48:46', 'female', 'Disease F'), ('14:15:38', 'female', 'Disease H'), ('14:42:34', 'female', 'Disease G'), ('15:18:26', 'male', 'Disease D'), ('16:00:25', 'male', 'Disease C'), ('16:17:08', 'female', 'Disease F'), ('16:32:36', 'male', 'Disease C'), ('16:54:19', 'male', 'Disease D'), ('17:26:49', 'female', 'Disease G'), ('17:50:13', 'female', 'Disease I'), ('17:53:38', 'female', 'Disease G'), ('17:57:25', 'male', 'Disease C'), ('18:07:18', 'male', 'Disease D'), ('18:38:35', 'male', 'Disease B'), ('18:44:22', 'male', 'Disease 